# MoE Experts: From Scratch

Now that we have a router, we need to actually route the tokens to the experts. This is the most complex part of implementing MoE.

In this tutorial, we will:
1. Create the Expert layer (FFN).
2. Implement the routing logic manually (slow but easy to understand).
3. See how to do it efficiently with masks.

## 1. The Expert

An expert is just a standard FeedForward Network (FFN).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.SiLU(),
            nn.Linear(d_ff, d_model, bias=False)
        )

    def forward(self, x):
        return self.net(x)

## 2. Manual Routing (The Loop)

Conceptually, we iterate over every token, check which experts it needs, run them, and sum the results. This is slow but correct.

In [ ]:
def manual_moe_forward(x, router, experts):
    # x: [batch, seq, d_model]
    batch_size, seq_len, d_model = x.shape
    output = torch.zeros_like(x)
    
    # Get routing decisions
    weights, indices = router(x)
    
    for b in range(batch_size):
        for t in range(seq_len):
            # For each selected expert (e.g., top-2)
            for k in range(indices.size(-1)):
                expert_idx = indices[b, t, k]
                weight = weights[b, t, k]
                
                # Get the token
                token = x[b, t].unsqueeze(0) # [1, d_model]
                
                # Run the expert
                expert_out = experts[expert_idx](token)
                
                # Add to output
                output[b, t] += weight * expert_out.squeeze(0)
                
    return output

## 3. Efficient Routing (Masking)

In practice, we want to process all tokens for a given expert at once (batching). We can do this by creating a mask for each expert.

In [ ]:
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_experts)])
        # Assuming TopKRouter is defined from previous lesson
        # self.router = TopKRouter(d_model, num_experts, top_k)

    def forward(self, x, weights, indices):
        output = torch.zeros_like(x)
        
        # Iterate over experts (not tokens!)
        for i in range(self.num_experts):
            # Find which tokens send to this expert
            # indices shape: [batch, seq, top_k]
            mask = (indices == i).any(dim=-1)
            
            if mask.any():
                # Select tokens
                selected_tokens = x[mask]
                
                # Run expert
                expert_out = self.experts[i](selected_tokens)
                
                # We need to apply the correct weight for each token
                # This part is tricky: we need to find WHICH of the top-k weights corresponds to this expert
                # For simplicity in this tutorial, we'll skip the complex gathering logic
                # and assume a simplified weighting for demonstration
                
                # Add to output (simplified)
                output[mask] += expert_out
                
        return output

The full implementation in `models/components.py` handles the weighting correctly using `gather` operations. It's a bit more complex but follows this same principle of masking.